In [1]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

Welcome to JupyROOT 6.18/00


In [2]:
from __future__ import print_function, division
import sys
import os
import rhalphalib as rl
import numpy as np
import scipy.stats
import pickle
import ROOT
rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

In [3]:
def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z+jets"] = ("Z+*",)
#     bkg_map["Z+HF"] = ("Z+HF",)
#     bkg_map["Z+LF"] = ("Z+LF",)
#     bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    
#     haven't processed signal yet
#     for signal in hists['sig']['template'].identifiers('process'):
#         if 'mhs' not in str(signal): continue
#         sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
#         signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT binning',bkg_hists["template"].axis("mT").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin(
        "mT", hist.Bin("mT", "mT", mT_binning)
    )
#     signal_hists["template"] = signal_hists["template"].rebin(
#         "mT", hist.Bin("mT", "mT", recoil_binning)
#     )
    data_hists["template"] = data_hists["template"].rebin(
        "mT", hist.Bin("mT", "mT", mT_binning)
    )

    hists = {"bkg": bkg_hists, "data": data_hists}

    return hists

In [4]:
np.linspace(0,600,10)

array([  0.        ,  66.66666667, 133.33333333, 200.        ,
       266.66666667, 333.33333333, 400.        , 466.66666667,
       533.33333333, 600.        ])

In [5]:
year = '2018'

mT_binning_dict = {
    "2018": np.linspace(0,600,21),
    "2017": np.linspace(0,600,21),
    "2016": np.linspace(0,600,21)
}
mT_binning = mT_binning_dict[year]
mTbins = np.array(mT_binning)
nmT = len(mTbins) - 1

###
# Extract histograms from input file
###

print("Grouping histograms")
hists = load("../hists/UL_lep_monotop_v1_test_" + year + ".scaled")
hists = remap_histograms(hists)
data_hists = hists["data"]
bkg_hists = hists["bkg"]
# signal_hists = hists["sig"]

###
# Preparing histograms for fit
##

data = {}
for r in data_hists["template"].identifiers("region"):
    data[str(r)] = data_hists["template"].integrate("region", r)

background = {}
for r in bkg_hists["template"].identifiers("region"):
    background[str(r)] = bkg_hists["template"].integrate("region", r)


Grouping histograms
initial mT binning [  0.  30.  60.  90. 120. 150. 180. 210. 240. 270. 300. 330. 360. 390.
 420. 450. 480. 510. 540. 570. 600.]


In [6]:
# channels are control regions
model = rl.Model("testMonojet")
mT = rl.Observable('mT', mTbins)

weCh = rl.Channel("weCh")
model.addChannel(weCh)

wmCh = rl.Channel("wmCh")
model.addChannel(wmCh)

<Model (testMonojet) instance at 0x7f3972fc5da0>

In [7]:
import scipy
def expo_sample(norm, scale, obs):
    cdf = scipy.stats.expon.cdf(scale=scale, x=obs.binning) * norm
    return (np.diff(cdf), obs.binning, obs.name)

In [8]:
model = rl.Model("testMonojet")
recoilbins = np.linspace(300, 1200, 13)
recoil = rl.Observable('recoil', recoilbins)
gammaCh = rl.Channel("gammaCh")
model.addChannel(gammaCh)

gammaTemplate = expo_sample(2000, 450, recoil)
gammaJetsMC = rl.TemplateSample('gammaJetsMC', rl.Sample.BACKGROUND, gammaTemplate)
# gammaJetsMC.setParamEffect(jec, np.random.normal(loc=1, scale=0.05, size=recoil.nbins))
# gammaJetsMC.setParamEffect(pho_id_eff, np.random.normal(loc=1, scale=0.02, size=recoil.nbins))

In [9]:
zllTemplate

NameError: name 'zllTemplate' is not defined

In [10]:
###
###
# Setting up systematics
###
###
lumi = rl.NuisanceParameter("lumi" + year, "lnN")
zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
id_e = rl.NuisanceParameter("id_e" + year, "lnN")
id_mu = rl.NuisanceParameter("id_mu" + year, "lnN")
id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
reco_e = rl.NuisanceParameter("reco_e" + year, "lnN")
iso_mu = rl.NuisanceParameter("iso_mu" + year, "lnN")
trig_e = rl.NuisanceParameter("trig_e" + year, "lnN")
trig_met = rl.NuisanceParameter("trig_met" + year, "lnN")
trig_pho = rl.NuisanceParameter("trig_pho" + year, "lnN")
veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
jec = rl.NuisanceParameter("jec" + year, "lnN")
#btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
ew1 = rl.NuisanceParameter("ew1", "lnN")
#ew2G = rl.NuisanceParameter("ew2G", "lnN")
ew2W = rl.NuisanceParameter("ew2W", "lnN")
ew2Z = rl.NuisanceParameter("ew2Z", "lnN")
#ew3G = rl.NuisanceParameter("ew3G", "lnN")
ew3W = rl.NuisanceParameter("ew3W", "lnN")
ew3Z = rl.NuisanceParameter("ew3Z", "lnN")
mix = rl.NuisanceParameter("mix", "lnN")
#muF = rl.NuisanceParameter("muF", "lnN")
#muR = rl.NuisanceParameter("muR", "lnN")
qcd1 = rl.NuisanceParameter("qcd1", "lnN")
qcd2 = rl.NuisanceParameter("qcd2", "lnN")
qcd3 = rl.NuisanceParameter("qcd3", "lnN")
whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
#ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")

In [11]:
def addBtagSyst(dictionary, process, region, templ):
    btagUp = template(dictionary, process, "btagUp", region)[0]
    btagDown = template(dictionary, process, "btagDown", region)[0]
    templ.setParamEffect(btag, btagUp, btagDown)

In [12]:
def addVJetsSyst(dictionary, process, region, templ):
    def addSyst(dictionary, process, region, templ, syst, string):
        histogram = dictionary[region].integrate("process", process)
        nominal=histogram.integrate("systematic", "nominal").values()[()]
        up=histogram.integrate("systematic", string+"Up").values()[()]
        down=histogram.integrate("systematic",string+"Down").values()[()]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, process, region, templ, ew1, "ew1")
    addSyst(dictionary, process, region, templ, ew2W, "ew2W")
    addSyst(dictionary, process, region, templ, ew2Z, "ew2Z")
    addSyst(dictionary, process, region, templ, ew3W, "ew3W")
    addSyst(dictionary, process, region, templ, ew3Z, "ew3Z")
    addSyst(dictionary, process, region, templ, mix, "mix")
    addSyst(dictionary, process, region, templ, qcd1, "qcd1")
    addSyst(dictionary, process, region, templ, qcd2, "qcd2")
    addSyst(dictionary, process, region, templ, qcd3, "qcd3")

In [13]:
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process)
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT", sumw2)
    return (output, binning, "mT")

In [14]:
template(dictionary, process, "btagUp", region)

NameError: name 'dictionary' is not defined

In [15]:
histogram = background['wjete'].integrate("process", 'W+jets')
nominal=histogram.integrate("systematic", "nominal").values()[()]
up=histogram.integrate("systematic", "ew1Up").values()[()]

KeyError: ()

In [16]:
histogram.identifiers('systematic')

[<StringBin (btagDown) instance at 0x7f3995a64470>,
 <StringBin (btagUp) instance at 0x7f3995a64438>,
 <StringBin (nominal) instance at 0x7f3995a643c8>]

In [38]:
background['wjete'].identifiers('process')

[<StringBin (DY+jets) instance at 0x7f6c3801bc50>,
 <StringBin (VV) instance at 0x7f6c3801bcc0>,
 <StringBin (ST) instance at 0x7f6c18669898>,
 <StringBin (TT) instance at 0x7f6c3801bd68>,
 <StringBin (W+jets) instance at 0x7f6c18669860>,
 <StringBin (QCD) instance at 0x7f6c1868abe0>]

In [56]:
template(data, "EGamma", "data", "wjete")

(array([0.00000e+00, 8.21950e+04, 2.86399e+05, 1.69375e+05, 6.19680e+04,
        2.17540e+04, 9.26700e+03, 4.91300e+03, 2.78900e+03, 1.66900e+03,
        1.09400e+03, 6.91000e+02, 5.03000e+02, 3.07000e+02, 2.46000e+02,
        1.93000e+02, 1.43000e+02, 1.08000e+02, 8.50000e+01, 6.70000e+01]),
 array([  0.,  30.,  60.,  90., 120., 150., 180., 210., 240., 270., 300.,
        330., 360., 390., 420., 450., 480., 510., 540., 570., 600.]),
 'mT')

In [57]:
wecr_wjetsTemplate = template(background, "W+jets", "nominal", "wjete", read_sumw2=True)
wecr_wjetsTemplate

(array([0.00000000e+00, 9.56082672e+04, 3.48076707e+05, 1.77362568e+05,
        5.59540697e+04, 1.93321656e+04, 9.54921853e+03, 5.09419732e+03,
        3.18622275e+03, 2.46089953e+03, 1.61558163e+03, 9.62954013e+02,
        7.77096521e+02, 4.76544882e+02, 3.16113002e+02, 2.29732455e+02,
        2.38579024e+02, 1.85212735e+02, 1.05765982e+02, 1.35737795e+02]),
 array([  0.,  30.,  60.,  90., 120., 150., 180., 210., 240., 270., 300.,
        330., 360., 390., 420., 450., 480., 510., 540., 570., 600.]),
 'mT',
 array([0.00000000e+00, 1.53603561e+06, 1.17269186e+07, 2.42312833e+06,
        8.19522215e+05, 2.45170518e+05, 1.21599714e+05, 7.60009599e+04,
        3.81491000e+04, 1.99368428e+05, 1.91912282e+04, 1.09982051e+04,
        9.21898897e+03, 5.95837008e+03, 4.10203753e+03, 2.88093143e+03,
        2.26872436e+03, 3.02264440e+03, 1.77875379e+03, 1.73389332e+03]))

In [10]:
wecr_wjetsMC = rl.TemplateSample(
    "wecr" + model_id + "_wjetsMC",
    rl.Sample.BACKGROUND,
    wecr_wjetsTemplate
)
type(wecr_wjetsMC)

NameError: name 'model_id' is not defined

In [17]:
wjete_wjetsMC

NameError: name 'wjete_wjetsMC' is not defined

# Wecr

In [18]:
year='2018'
model_id = 'ControlRegionModel'
model = rl.Model(model_id)
ch_name = "wjete" 
# Create a channel/Control region
wjete = rl.Channel(ch_name)
# add the Channel/CR to the model
model.addChannel(wjete)

###
# Add data distribution to the channel
###

if year == "2018":
    wjete.setObservation(template(data, "EGamma", "data", "wjete"))
else:
    wjete.setObservation(template(data, "SingleElectron", "data","wjete"))

###
# W(->lnu)+jets data-driven model
###

wjete_wjetsTemplate = template(background, "W+jets", "nominal", "wjete", read_sumw2=True)
wjete_wjetsMC = rl.TemplateSample(
    "wjete_" + model_id + "_wjetsMC",
    rl.Sample.BACKGROUND,
    wjete_wjetsTemplate
)
wjete_wjetsMC.setParamEffect(lumi, 1.027)
wjete_wjetsMC.setParamEffect(trig_e, 1.01)
wjete_wjetsMC.setParamEffect(veto_tau, 1.03)
wjete_wjetsMC.setParamEffect(wjets_norm, 1.4)
wjete_wjetsMC.setParamEffect(jec, 1.05)
wjete_wjetsMC.setParamEffect(id_e, 1.02)
wjete_wjetsMC.setParamEffect(reco_e, 1.02)
#wecr_wjetsMC.autoMCStats()
addBtagSyst(background, "W+jets", "wjete", wjete_wjetsMC)
# addVJetsSyst(background, "W+jets", "wjete", wecr_wjetsMC)

# wecr_wjetsTFstatParameters =  np.array([rl.NuisanceParameter("wecr_"+year+"_wjetsTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(wecr_wjetsTemplate[0].size)])
# wecr_wjetsTransferFactor = wecr_wjetsMC.getExpectation() / sr_wjetsMC.getExpectation()
# nominal =  wecr_wjetsTemplate[0] / sr_wjetsTemplate[0]
# dz = simple_error_propagation(sr_wjetsTemplate[0], wecr_wjetsTemplate[0], sr_wjetsTemplate[3], wecr_wjetsTemplate[3])
# wecr_wjetsTransferFactor = wecr_wjetsTransferFactor * ( 1. + (dz/nominal)*wecr_wjetsTFstatParameters )
# wecr_wjets = rl.TransferFactorSample(
#     ch_name + "_wjets", rl.Sample.BACKGROUND, wecr_wjetsTransferFactor, sr_wjets
# )
wjete.addSample(wjete_wjetsMC)

# ###
# # top-antitop model
# ###

# wecr_ttTemplate = template(background, "TT", "nominal","wjete", read_sumw2=True)
# wecr_ttMC = rl.TemplateSample(
#     "wecr" + model_id + "_ttMC",
#     rl.Sample.BACKGROUND,
#     wecr_ttTemplate
# )
# wecr_ttMC.setParamEffect(lumi, 1.027)
# wecr_ttMC.setParamEffect(trig_e, 1.01)
# wecr_ttMC.setParamEffect(veto_tau, 1.03)
# wecr_ttMC.setParamEffect(jec, 1.05)
# wecr_ttMC.setParamEffect(id_e, 1.02)
# wecr_ttMC.setParamEffect(reco_e, 1.02)
# addBtagSyst(background, recoil, "TT", "wecr", wecr_ttMC, category)

# if category == "pass":
#     wecr_ttMC.setParamEffect(tt_norm, 1.2)
#     #wecr_ttMC.autoMCStats()
#     wecr_ttTFstatParameters =  np.array([rl.NuisanceParameter("wecr_"+year+"_ttTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(wecr_ttTemplate[0].size)])
#     wecr_ttTransferFactor = wecr_ttMC.getExpectation() / sr_ttMC.getExpectation()
#     nominal =  wecr_ttTemplate[0] / sr_ttTemplate[0]
#     dz = simple_error_propagation(sr_ttTemplate[0], wecr_ttTemplate[0], sr_ttTemplate[3], wecr_ttTemplate[3])
#     wecr_ttTransferFactor = wecr_wjetsTransferFactor * ( 1. + (dz/nominal)*wecr_ttTFstatParameters )
#     wecr_tt = rl.TransferFactorSample(
#         ch_name + "_tt", rl.Sample.BACKGROUND, wecr_ttTransferFactor, sr_tt
#     )
#     wecr.addSample(wecr_tt)
# else:
#     wecr_ttMC.setParamEffect(ttMC_norm, 1.2)
#     wecr.addSample(wecr_ttMC)

# ###
# # Other MC-driven processes
# ###

# wecr_stTemplate = template(background, "ST", "nominal", recoil, "wecr", category)
# wecr_st = rl.TemplateSample(
#     ch_name + "_stMC", rl.Sample.BACKGROUND, wecr_stTemplate
# )
# wecr_st.setParamEffect(lumi, 1.027)
# wecr_st.setParamEffect(trig_e, 1.01)
# wecr_st.setParamEffect(veto_tau, 1.03)
# wecr_st.setParamEffect(st_norm, 1.2)
# wecr_st.setParamEffect(jec, 1.05)
# wecr_st.setParamEffect(id_e, 1.02)
# wecr_st.setParamEffect(reco_e, 1.02)
# addBtagSyst(background, recoilbin, "ST", "wecr", wecr_st, category)
# wecr.addSample(wecr_st)

# wecr_dyjetsTemplate = template(background, "DY+jets", "nominal", recoil, "wecr", category)
# wecr_dyjets = rl.TemplateSample(
#     ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, wecr_dyjetsTemplate
# )
# wecr_dyjets.setParamEffect(lumi, 1.027)
# wecr_dyjets.setParamEffect(trig_e, 1.01)
# wecr_dyjets.setParamEffect(veto_tau, 1.03)
# wecr_dyjets.setParamEffect(zjetsMC_norm, 1.4)
# wecr_dyjets.setParamEffect(jec, 1.05)
# wecr_dyjets.setParamEffect(id_e, 1.02)
# wecr_dyjets.setParamEffect(reco_e, 1.02)
# addBtagSyst(background, recoilbin, "DY+jets", "wecr", wecr_dyjets, category)
# addVJetsSyst(background, recoil, "DY+jets", "wecr", wecr_dyjets, category)
# wecr.addSample(wecr_dyjets)

# wecr_vvTemplate = template(background, "VV", "nominal", recoil, "wecr", category)
# wecr_vv = rl.TemplateSample(
#     ch_name + "_vvMC", rl.Sample.BACKGROUND, wecr_vvTemplate
# )
# wecr_vv.setParamEffect(lumi, 1.027)
# wecr_vv.setParamEffect(trig_e, 1.01)
# wecr_vv.setParamEffect(veto_tau, 1.03)
# wecr_vv.setParamEffect(vv_norm, 1.2)
# wecr_vv.setParamEffect(jec, 1.05)
# wecr_vv.setParamEffect(id_e, 1.02)
# wecr_vv.setParamEffect(reco_e, 1.02)
# addBtagSyst(background, recoilbin, "VV", "wecr", wecr_vv, category)
# wecr.addSample(wecr_vv)

# wecr_hbbTemplate = template(background, "Hbb", "nominal", recoil, "wecr", category)
# wecr_hbb = rl.TemplateSample(
#     ch_name + "_hbbMC", rl.Sample.BACKGROUND, wecr_hbbTemplate
# )
# wecr_hbb.setParamEffect(lumi, 1.027)
# wecr_hbb.setParamEffect(trig_e, 1.01)
# wecr_hbb.setParamEffect(veto_tau, 1.03)
# wecr_hbb.setParamEffect(hbb_norm, 1.2)
# wecr_hbb.setParamEffect(jec, 1.05)
# wecr_hbb.setParamEffect(id_e, 1.02)
# wecr_hbb.setParamEffect(reco_e, 1.02)
# addBtagSyst(background, recoilbin, "Hbb", "wecr", wecr_hbb, category)
# wecr.addSample(wecr_hbb)

# wecr_qcdTemplate = template(background, "QCD", "nominal", recoil, "wecr", category)
# wecr_qcd = rl.TemplateSample(
#     ch_name + "_qcdMC", rl.Sample.BACKGROUND, wecr_qcdTemplate
# )
# wecr_qcd.setParamEffect(lumi, 1.027)
# wecr_qcd.setParamEffect(trig_e, 1.01)
# wecr_qcd.setParamEffect(veto_tau, 1.03)
# wecr_qcd.setParamEffect(qcde_norm, 2.0)
# wecr_qcd.setParamEffect(jec, 1.05)
# wecr_qcd.setParamEffect(id_e, 1.02)
# wecr_qcd.setParamEffect(reco_e, 1.02)
# addBtagSyst(background, recoilbin, "QCD", "wecr", wecr_qcd, category)
# wecr.addSample(wecr_qcd)

In [20]:
wjete_wjetsMC.show()

[0.00000000e+00 9.56082672e+04 3.48076707e+05 1.77362568e+05
 5.59540697e+04 1.93321656e+04 9.54921853e+03 5.09419732e+03
 3.18622275e+03 2.46089953e+03 1.61558163e+03 9.62954013e+02
 7.77096521e+02 4.76544882e+02 3.16113002e+02 2.29732455e+02
 2.38579024e+02 1.85212735e+02 1.05765982e+02 1.35737795e+02]
[0.00000000e+00 1.53603561e+06 1.17269186e+07 2.42312833e+06
 8.19522215e+05 2.45170518e+05 1.21599714e+05 7.60009599e+04
 3.81491000e+04 1.99368428e+05 1.91912282e+04 1.09982051e+04
 9.21898897e+03 5.95837008e+03 4.10203753e+03 2.88093143e+03
 2.26872436e+03 3.02264440e+03 1.77875379e+03 1.73389332e+03]


In [21]:
s[:s.find('_')]

'wjete'